In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_excel(r"C:\Pan card\PAN Number Validation Dataset.xlsx")

In [3]:
df.head(10)

,Pan_Numbers
0,VGLOD3180G
1,PHOXD7232L
2,MGEPH6532A
3,JJCHK4574O
4,XTQIJ2330L
5,HTJYM3835H
6,YQTAP6661X
7,hvofe5635y
8,hyuij7902r
9,idsmt3429e


In [4]:
len(df)

10000

In [5]:
total_records = len(df)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Pan_Numbers  9035 non-null   object
dtypes: object(1)
memory usage: 78.3+ KB


### DATA CLEANING

In [7]:
df["Pan_Numbers"] = df["Pan_Numbers"].astype("string")

In [8]:
df["Pan_Numbers"].str.strip()

0       VGLOD3180G
1       PHOXD7232L
2       MGEPH6532A
3       JJCHK4574O
4       XTQIJ2330L
           ...    
9995    TNGUY429!V
9996    SMMN33673g
9997          <NA>
9998    RLAI69795t
9999       YWY2482
Name: Pan_Numbers, Length: 10000, dtype: string

In [9]:
df["Pan_Numbers"].str.upper()

0       VGLOD3180G
1       PHOXD7232L
2       MGEPH6532A
3       JJCHK4574O
4       XTQIJ2330L
           ...    
9995    TNGUY429!V
9996    SMMN33673G
9997          <NA>
9998    RLAI69795T
9999       YWY2482
Name: Pan_Numbers, Length: 10000, dtype: string

In [10]:
print(df[df["Pan_Numbers"]==''])

Empty DataFrame
Columns: [Pan_Numbers]
Index: []


In [11]:
print(df[df["Pan_Numbers"].isna()])

     Pan_Numbers
5022        <NA>
5027        <NA>
5033        <NA>
5043        <NA>
5057        <NA>
...          ...
9961        <NA>
9972        <NA>
9986        <NA>
9987        <NA>
9997        <NA>

[965 rows x 1 columns]


In [12]:
df = df.replace({"Pan_Numbbers":""},pd.NA)

In [13]:
df = df.dropna(subset="Pan_Numbers")

In [14]:
len(df)

9035

In [15]:
df["Pan_Numbers"].nunique()

9027

In [16]:
df = df.drop_duplicates(subset= "Pan_Numbers", keep = "first")

In [17]:
len(df)

9027

### VALIDATION

In [18]:
def has_adjacent_rep(pan):
    return any(pan[i] == pan[i+1] for i in range(len(pan)-1))

In [19]:
def is_sequential(pan):
    return all(ord(pan[i+1]) - ord(pan[i]) ==1 for i in range(len(pan)-1))

In [20]:
def is_valid_pan(pan):
    if len(pan) != 10:
        return False

    if not re.match(r'^[A-Z]{5}[0-9]{4}[A-Z]$', pan):
        return False

    if has_adjacent_rep(pan):
        return False

    if is_sequential(pan):
        return False

    return True

In [21]:
df["Status"] = df["Pan_Numbers"].apply(lambda x: "Valid" if is_valid_pan(x) else "Invalid")

In [22]:
print(df.head(10))

    Pan_Numbers   Status
0    VGLOD3180G    Valid
1    PHOXD7232L    Valid
2    MGEPH6532A    Valid
3    JJCHK4574O  Invalid
4    XTQIJ2330L  Invalid
5    HTJYM3835H    Valid
6    YQTAP6661X  Invalid
7    hvofe5635y  Invalid
8  hyuij7902r    Invalid
9    idsmt3429e  Invalid


In [23]:
valid_cnt = (df["Status"] == "Valid").sum()

In [24]:
invalid_cnt = (df["Status"] == "Invalid").sum()

In [25]:
missing_cnt = total_records - (valid_cnt+invalid_cnt)

In [26]:
print(total_records)

10000


In [27]:
print(valid_cnt)

3187


In [28]:
print(invalid_cnt)

5840


In [29]:
print(missing_cnt)

973


In [30]:
df_summary = pd.DataFrame({"Total Processed Records":[total_records], "Total Valid count":[valid_cnt], "Total Invalid count":[invalid_cnt], "Total Missing Count":[missing_cnt]})

In [31]:
print(df_summary.head())

   Total Processed Records  Total Valid count  Total Invalid count  \
0                    10000               3187                 5840   

   Total Missing Count  
0                  973  


In [32]:
with pd.ExcelWriter(r"C:\Pan card\Pan Validation Result.xlsx") as writer:
    df.to_excel(writer, sheet_name="Pan Validations", index=False)
    df_summary.to_excel(writer, sheet_name="Summary", index=False)
